# SA Artists Multi-Source Data Lake
Combining **YouTube**, **Last.fm** and **MusicBrainz** to analyze top SA Amapiano & Afrobeats artists.

| Source | Data |
|--------|------|
| YouTube API | Videos, views, likes, comments |
| Last.fm API | Play counts, listeners, top tracks |
| MusicBrainz | Artist metadata, genres, country |

## 1️ Install Dependencies

In [ ]:
!pip install google-api-python-client pandas matplotlib seaborn requests -q

## 2️ Load API Keys from Colab Secrets
Add `YOUTUBE_API_KEY` and `LASTFM_API_KEY` to Colab Secrets ( icon on left sidebar)

In [ ]:
from google.colab import userdata

try:
    YOUTUBE_API_KEY = userdata.get('YOUTUBE_API_KEY')
    print(' YouTube API Key loaded')
except:
    YOUTUBE_API_KEY = 'your_youtube_api_key_here'
    print('  Add YOUTUBE_API_KEY to Colab Secrets')

try:
    LASTFM_API_KEY = userdata.get('LASTFM_API_KEY')
    print(' Last.fm API Key loaded')
except:
    LASTFM_API_KEY = 'your_lastfm_api_key_here'
    print('  Add LASTFM_API_KEY to Colab Secrets')

## 3️ Artist Config

In [ ]:
ARTISTS = [
    'Kabza De Small',
    'DJ Maphorisa',
    'Burna Boy',
    'Davido',
    'Masters KG',
]

YOUTUBE_CHANNEL_IDS = {
    'Kabza De Small': 'UCxoVP9cGJHBZaLMexO0fCkw',
    'DJ Maphorisa':   'UCFXyVm03fMlzI4B4sQkDtBA',
    'Burna Boy':      'UCEzDdNqNkT-7rSfSGSr1hWg',
    'Davido':         'UCkBV3nBa0iRdxEGc4DUS3xA',
    'Masters KG':     'UC-Vdvbvw_57gspD6A2m4vkw',
}
print(' Config loaded')

## 4️ Fetch YouTube Data

In [ ]:
from googleapiclient.discovery import build
import pandas as pd

def fetch_youtube_videos(artist_name, channel_id, max_results=50):
    youtube = build('youtube', 'v3', developerKey=YOUTUBE_API_KEY)
    videos = []
    try:
        response = youtube.channels().list(part='contentDetails', id=channel_id).execute()
        uploads  = response['items'][0]['contentDetails']['relatedPlaylists']['uploads']
        playlist = youtube.playlistItems().list(
            part='contentDetails', playlistId=uploads, maxResults=max_results
        ).execute()
        video_ids = [i['contentDetails']['videoId'] for i in playlist['items']]
        stats = youtube.videos().list(part='snippet,statistics', id=','.join(video_ids)).execute()
        for item in stats['items']:
            s = item.get('statistics', {})
            videos.append({
                'artist_name':   artist_name,
                'title':         item['snippet']['title'],
                'published_at':  item['snippet']['publishedAt'],
                'view_count':    int(s.get('viewCount', 0)),
                'like_count':    int(s.get('likeCount', 0)),
                'comment_count': int(s.get('commentCount', 0)),
            })
    except Exception as e:
        print(f'   YouTube error: {e}')
    return videos

youtube_data = []
for artist, channel_id in YOUTUBE_CHANNEL_IDS.items():
    print(f' Fetching YouTube: {artist}...')
    videos = fetch_youtube_videos(artist, channel_id)
    youtube_data.extend(videos)
    print(f'    {len(videos)} videos')

df_youtube = pd.DataFrame(youtube_data)
df_youtube['published_at'] = pd.to_datetime(df_youtube['published_at'])
df_youtube['engagement_rate'] = ((df_youtube['like_count'] + df_youtube['comment_count']) / df_youtube['view_count'] * 100).round(2)
print(f'\n Total YouTube videos: {len(df_youtube)}')
df_youtube.head()

## 5️ Fetch Last.fm Data

In [ ]:
import requests

def fetch_lastfm_artist(artist_name):
    url = 'http://ws.audioscrobbler.com/2.0/'
    try:
        r = requests.get(url, params={
            'method': 'artist.getinfo', 'artist': artist_name,
            'api_key': LASTFM_API_KEY, 'format': 'json'
        })
        data  = r.json().get('artist', {})
        stats = data.get('stats', {})
        return {
            'artist_name': artist_name,
            'listeners':   int(stats.get('listeners', 0)),
            'play_count':  int(stats.get('playcount', 0)),
            'tags':        ', '.join([t['name'] for t in data.get('tags', {}).get('tag', [])[:5]]),
        }
    except Exception as e:
        print(f'   Last.fm error: {e}')
        return None

def fetch_lastfm_top_tracks(artist_name, limit=10):
    url = 'http://ws.audioscrobbler.com/2.0/'
    tracks = []
    try:
        r = requests.get(url, params={
            'method': 'artist.gettoptracks', 'artist': artist_name,
            'api_key': LASTFM_API_KEY, 'format': 'json', 'limit': limit
        })
        for t in r.json().get('toptracks', {}).get('track', []):
            tracks.append({
                'artist_name': artist_name,
                'track_name':  t['name'],
                'play_count':  int(t.get('playcount', 0)),
                'listeners':   int(t.get('listeners', 0)),
                'rank':        int(t['@attr']['rank']),
            })
    except Exception as e:
        print(f'   Last.fm tracks error: {e}')
    return tracks

lastfm_artists = []
lastfm_tracks  = []
for artist in ARTISTS:
    print(f' Fetching Last.fm: {artist}...')
    info = fetch_lastfm_artist(artist)
    if info:
        lastfm_artists.append(info)
        print(f'    {info["listeners"]:,} listeners, {info["play_count"]:,} plays')
    tracks = fetch_lastfm_top_tracks(artist)
    lastfm_tracks.extend(tracks)

df_lastfm  = pd.DataFrame(lastfm_artists)
df_tracks  = pd.DataFrame(lastfm_tracks)
print(f'\n Last.fm artists: {len(df_lastfm)}')
df_lastfm

## 6️ Fetch MusicBrainz Data

In [ ]:
import time

def fetch_musicbrainz(artist_name):
    headers = {'User-Agent': 'sa-artists-analytics/1.0'}
    try:
        r = requests.get('https://musicbrainz.org/ws/2/artist/', params={
            'query': f'artist:{artist_name}', 'fmt': 'json', 'limit': 1
        }, headers=headers)
        artists = r.json().get('artists', [])
        if not artists:
            return None
        a    = artists[0]
        tags = [t['name'] for t in a.get('tags', [])[:5]]
        return {
            'artist_name': artist_name,
            'country':     a.get('area', {}).get('name', 'Unknown'),
            'type':        a.get('type', 'Unknown'),
            'begin_date':  a.get('life-span', {}).get('begin', 'Unknown'),
            'genres':      ', '.join(tags),
        }
    except Exception as e:
        print(f'   MusicBrainz error: {e}')
        return None
    finally:
        time.sleep(1)

mb_data = []
for artist in ARTISTS:
    print(f' Fetching MusicBrainz: {artist}...')
    data = fetch_musicbrainz(artist)
    if data:
        mb_data.append(data)
        print(f'    {data["country"]} | {data["genres"]}')

df_musicbrainz = pd.DataFrame(mb_data)
print(f'\n MusicBrainz artists: {len(df_musicbrainz)}')
df_musicbrainz

## 7️ Join All Sources Together

In [ ]:
# YouTube summary per artist
yt_summary = df_youtube.groupby('artist_name').agg(
    total_videos    = ('title', 'count'),
    total_yt_views  = ('view_count', 'sum'),
    total_likes     = ('like_count', 'sum'),
    avg_engagement  = ('engagement_rate', 'mean'),
).reset_index()

# Join all sources on artist_name
df_combined = yt_summary \
    .merge(df_lastfm, on='artist_name', how='left') \
    .merge(df_musicbrainz, on='artist_name', how='left')

# Derived metrics
df_combined['views_per_listener'] = (df_combined['total_yt_views'] / df_combined['listeners']).round(2)
df_combined['total_yt_views_M']   = (df_combined['total_yt_views'] / 1e6).round(2)
df_combined['play_count_M']       = (df_combined['play_count'] / 1e6).round(2)
df_combined['listeners_M']        = (df_combined['listeners'] / 1e6).round(2)

print(' Combined dataset:')
df_combined[['artist_name', 'total_videos', 'total_yt_views_M', 'listeners_M', 'play_count_M', 'country', 'genres']]

## 8️ Visualizations

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

sns.set_theme(style='darkgrid')
COLORS = ['#FF6B6B', '#FFD93D', '#6BCB77', '#4D96FF', '#C77DFF']

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle(' SA Artists — Multi-Source Analytics Dashboard', fontsize=18, fontweight='bold')

# Chart 1: YouTube Views
ax1 = axes[0, 0]
ax1.barh(df_combined['artist_name'], df_combined['total_yt_views_M'], color=COLORS)
ax1.set_title('YouTube Total Views (M)', fontweight='bold')
ax1.invert_yaxis()

# Chart 2: Last.fm Listeners
ax2 = axes[0, 1]
ax2.barh(df_combined['artist_name'], df_combined['listeners_M'], color=COLORS)
ax2.set_title('Last.fm Monthly Listeners (M)', fontweight='bold')
ax2.invert_yaxis()

# Chart 3: Last.fm Play Count
ax3 = axes[0, 2]
ax3.barh(df_combined['artist_name'], df_combined['play_count_M'], color=COLORS)
ax3.set_title('Last.fm Total Play Count (M)', fontweight='bold')
ax3.invert_yaxis()

# Chart 4: YouTube vs Last.fm scatter
ax4 = axes[1, 0]
for i, row in df_combined.iterrows():
    ax4.scatter(row['total_yt_views_M'], row['listeners_M'],
                color=COLORS[i % len(COLORS)], s=200, zorder=5)
    ax4.annotate(row['artist_name'], (row['total_yt_views_M'], row['listeners_M']),
                 textcoords='offset points', xytext=(5, 5), fontsize=8)
ax4.set_title('YouTube Views vs Last.fm Listeners', fontweight='bold')
ax4.set_xlabel('YouTube Views (M)')
ax4.set_ylabel('Last.fm Listeners (M)')

# Chart 5: Avg Engagement Rate
ax5 = axes[1, 1]
ax5.bar(df_combined['artist_name'], df_combined['avg_engagement'], color=COLORS)
ax5.set_title('Avg YouTube Engagement Rate (%)', fontweight='bold')
plt.setp(ax5.get_xticklabels(), rotation=20, ha='right')

# Chart 6: YouTube views over time
ax6 = axes[1, 2]
for i, (artist, group) in enumerate(df_youtube.groupby('artist_name')):
    monthly = group.resample('ME', on='published_at')['view_count'].sum().cumsum()
    ax6.plot(monthly.index, monthly / 1e6, marker='o', label=artist,
             color=COLORS[i % len(COLORS)], linewidth=2)
ax6.set_title('Cumulative YouTube Views Over Time', fontweight='bold')
ax6.set_ylabel('Views (M)')
ax6.legend(fontsize=7)
ax6.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.savefig('sa_artists_multi_source.png', dpi=150, bbox_inches='tight')
plt.show()
print(' Chart saved!')

## 9️ Save & Download Data

In [ ]:
from google.colab import files

df_youtube.to_csv('youtube_videos.csv', index=False)
df_lastfm.to_csv('lastfm_artists.csv', index=False)
df_tracks.to_csv('lastfm_tracks.csv', index=False)
df_musicbrainz.to_csv('musicbrainz_artists.csv', index=False)
df_combined.to_csv('combined_artists.csv', index=False)

print(' All CSVs saved!')
files.download('sa_artists_multi_source.png')
files.download('combined_artists.csv')